# 02   Parser Evaluation
**Project:** `energy-audit`

Compares 3 PDF-to-Markdown parsers **and** 2 LLM structure extractors against a
unified ground truth for the full EU energy-regulation corpus (25 docs).

**Pipeline position:** after `01` (corpus audit), before `03a` (structure
extraction) and `03b` (AST construction).

**Outputs:**
- `notebooks/data/ground_truth/ground_truth_all_docs.json`  (heuristic v1)
- `notebooks/data/ground_truth/ground_truth_all_docs_v2.json` (LLM-reviewed)
- `notebooks/data/evaluations/parser_evaluation_<ts>.csv`
- `notebooks/data/evaluations/best_parser.json`  (consumed by 03a / 03b)

**Design notes:**
- All LLM rows consume the **pymupdf4llm** markdown as input text, so models
  are compared on the same source.
- LLM calls are cached under `notebooks/data/llm_cache/` keyed by
  `(model, sha1(prompt))`   re-runs are free, rows are resumable.
- GT v1 = regex heuristic; **v2 = LLM-review corrected** (default reasoner).


## 0   Install dependencies

## 1   Imports & configuration

In [1]:
import os, re, sys, json, time, logging, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from rich.console import Console
from rich import print as rprint
from rich.progress import track

logging.basicConfig(level=logging.ERROR)
warnings.filterwarnings("ignore")
console = Console()

# --- repo root (same logic as src/common.py) ---
def _repo_root() -> Path:
    env = os.getenv("ENERGY_AUDIT_ROOT")
    if env and Path(env).exists():
        return Path(env).resolve()
    cur = Path.cwd().resolve()
    for cand in [cur, *cur.parents]:
        if (cand / "notebooks").is_dir() and (cand / "data").is_dir():
            return cand
    return cur.parent

ROOT = _repo_root()
sys.path.insert(0, str(ROOT / "src"))

import common as c            # paths, discovery, ollama, json
from parsing import parsers as P           # pdf -> markdown backends
from parsing import llm_extractor as LEX   # LLM structure extraction (cached)
from parsing import ground_truth as GT     # GT generation + LLM review
import time as _time

c.ensure_dirs()

# --- config ---
TEXT_PARSERS = ["pymupdf4llm", "docling", "unstructured"]
LLM_MODELS   = [c.env_models()["reasoner_default"],    # qwen3.8:27b
                c.env_models()["reasoner_baseline"]]    # deepseek-r1:32b

GT_MODE   = "v2"          # v2 | v1 | regen
N_WINDOWS = 6
EVAL_ALL  = True
SEED_DOCS = ["gdpr_2016_679", "dora_2022_2554", "remit_1227_2011"]

rprint("[bold]root :[/bold]", "../" + ROOT.name)
rprint("[bold]text par :[/bold]", TEXT_PARSERS)
rprint("[bold]llm rows :[/bold]", LLM_MODELS)
rprint("[bold]gt mode  :[/bold]", GT_MODE, f"({N_WINDOWS} windows/doc)")

root     : <repo root>

text par :
['pymupdf4llm', 'docling', 'unstructured']

llm rows :
['qwen3.8:27b', 'deepseek-r1:32b']

gt mode  : v2 (6 windows/doc)

## 2   Corpus status & PDF to Markdown conversion
Every PDF needs a `pymupdf4llm` markdown: it is the shared input text for all
LLM rows and the source for the text-parser rows. Missing ones are converted
on demand.

In [2]:
def doc_list():
    return c.discover_pdf()

def status_table() -> pd.DataFrame:
    rows = []
    for pdf in doc_list():
        md_path = c.md_for_pdf(pdf)
        rows.append({
            "filename": pdf.name,
            "doc_type": c.doc_type_for(pdf),
            "pdf_mb": round(pdf.stat().st_size / 1e6, 2),
            "md_ok": md_path.exists(),
        })
    return pd.DataFrame(rows).set_index("filename")

def convert_missing(parser: str = "pymupdf4llm") -> int:
    missing = [p for p in doc_list() if not c.md_for_pdf(p).exists()]
    if not missing:
        return 0
    rprint("[green]converting %d missing PDFs with %s ...[/green]" % (len(missing), parser))
    for pdf in missing:
        md_path = c.md_for_pdf(pdf)
        md_path.parent.mkdir(parents=True, exist_ok=True)
        md_path.write_text(P.PARSERS[parser](pdf), encoding="utf-8")
        print("  ok", pdf.name)
    return len(missing)

status = status_table()
console.print(status)
n_missing = int((~status["md_ok"]).sum())
if n_missing:
    convert_missing("pymupdf4llm")
n_ok = int(status["md_ok"].sum())
rprint("[green]ok %d / %d PDFs have markdown.[/green]" % (n_ok, len(status)))

doc_type  pdf_mb  md_ok
filename                                                       
data_act_2023_2854.pdf               regulations    1.41   True
dlt_pilot_2022_858.pdf               regulations    0.79   True
dora_2022_2554.pdf                   regulations    1.49   True
eidas_2014_910.pdf                   regulations    1.05   True
eidas_2_2024_1183.pdf                regulations    1.49   True
elec_reg_2019_943.pdf                regulations    0.92   True
emd_reform_reg_2024_1747.pdf         regulations    1.27   True
eu_ai_act_2024_1689.pdf              regulations    2.58   True
gdpr_2016_679.pdf                    regulations    0.98   True
metering_data_2023_1162.pdf          regulations    2.42   True
mica_2023_1114.pdf                   regulations    1.63   True
remit_1227_2011.pdf                  regulations    0.84   True
remit_ii_2024_1106.pdf               regulations    1.29   True
elec_dir_2019_944.pdf                 directives    0.96   True
emd_reform_dir_2024_1711.pdf          directives    1.09   True
eprivacy_dir_2002_58.pdf              directives    0.17   True
nis2_dir_2022_2555.pdf                directives    1.33   True
acer_remit_guidance.pdf                 guidance    2.18   True
entso-e_compliance_monitoring.pdf       guidance    0.63   True
entso-e_simulation_models.pdf           guidance    0.89   True
know_your_contract_guidance.pdf         guidance    0.47   True
entso_cacm_2015_1222.pdf           network_codes    0.68   True
entso_ebgl_2017_2195.pdf           network_codes    0.77   True
entso_ncrfg_2016_631.pdf           network_codes    1.03   True
entso_sogl_2017_1485.pdf           network_codes    1.04   True

ok 25 / 25 PDFs have markdown.

## 3   Ground truth (all docs, heuristic + LLM review)
Replaces the old 3-doc / page-split implementation. Per-doc schema:

| key | values |
|---|---|
| `global` | `document_title`, `instrument`, `celex`, `publication_date`, `article_numbers`, `obligations_count`, `tables_count` |
| `windows` | N sampled windows, each with `fields_found`, `tables`, `structure_elements` |
| `review`  | v2 only: `model`, `applied`, `changes[]` |

In [3]:
def get_ground_truth(mode: str = GT_MODE) -> dict:
    v2 = c.GROUND_TRUTH_PATH / "ground_truth_all_docs_v2.json"
    v1 = c.GROUND_TRUTH_PATH / "ground_truth_all_docs.json"
    if mode == "v2" and v2.exists():
        return json.loads(v2.read_text())
    if mode == "v1" and v1.exists():
        return json.loads(v1.read_text())

    g1 = GT.build_ground_truth(n_windows=N_WINDOWS)
    GT.save_ground_truth(g1, "ground_truth_all_docs.json")
    rprint("[green]heuristic GT: %d docs[/green]" % len(g1["documents"]))
    if mode in ("v2", "regen"):
        model = c.env_models()["reasoner_default"]
        g2 = GT.llm_review_ground_truth(g1, model=model)
        GT.save_ground_truth(g2, "ground_truth_all_docs_v2.json")
        rprint("[green]LLM-reviewed GT by %s done.[/green]" % model)
        return g2
    return g1

ground_truth = get_ground_truth()
rprint("[bold]GT docs :[/bold]", len(ground_truth["documents"]))
rprint("[bold]source  :[/bold]", ground_truth["_metadata"].get("generator"))

def selected_docs() -> dict:
    if EVAL_ALL:
        return ground_truth["documents"]
    return {k: v for k, v in ground_truth["documents"].items() if k in SEED_DOCS}

sel = selected_docs()
rprint("evaluating %d docs x %d text parsers + %d LLM rows"
       % (len(sel), len(TEXT_PARSERS), len(LLM_MODELS)))

GT docs : 25

source  : heuristic v2 (window sampling)

evaluating 25 docs x 3 text parsers + 2 LLM rows

## 4   Evaluator

One row per (doc, candidate). Text parsers and LLM extractors are scored on
the same ground-truth record:

| metric | range | how |
|---|---|---|
| `field_accuracy` | 0..1 | mean of title / instrument / publication-date hits vs `global` |
| `table_accuracy` | 0..1 | `tables_count` closeness to GT |
| `structure_score` | 0..1 | article-count closeness to GT |
| `obligation_match` | 0..1 | `obligations_count` closeness (LLM rows) |
| `processing_time_s` | s | wall time for the row |
| `error_rate` | 0/1 | 1 on exception or unparseable JSON |

Composite = mean(`field_accuracy`, `table_accuracy`, `structure_score`).


In [4]:

def _norm(t):
    if not t:
        return ""
    t = " ".join(str(t).replace("*", "").split()).lower()
    return t.replace("official journal of the european union", "").strip()

def _match(gt, got) -> int:
    """lenient equality: normalise whitespace and markdown, allow containment
    so a longer-but-correct title still scores."""
    if gt in (None, "", []):
        return 1
    a, b = _norm(gt), _norm(got)
    if not b:
        return 0
    return int(a == b or a in b or b in a)

def score_global(gt_global: dict, got: dict) -> dict:
    s = {
        "title_ok":      _match(gt_global.get("document_title"),  got.get("document_title")),
        "instrument_ok": _match(gt_global.get("instrument"),      got.get("instrument")),
    }
    pd_gt  = (gt_global.get("publication_date") or "")[:7]
    pd_got = (got.get("publication_date") or "")[:7]
    s["date_ok"] = 1 if (pd_gt and pd_got and pd_gt == pd_got) else (0 if pd_gt else 1)

    gt_t, got_t = int(gt_global.get("tables_count") or 0), int(got.get("tables_count") or 0)
    s["table_acc"] = round(min(1.0, abs(gt_t - got_t) / max(1, gt_t)), 3)

    gt_a  = len(gt_global.get("article_numbers") or [])
    got_a = len(got.get("articles") or [])
    s["structure"] = round(max(0.0, 1.0 - abs(gt_a - got_a) / max(1, gt_a)), 3)

    gt_o, got_o = int(gt_global.get("obligations_count") or 0), int(got.get("obligations_count") or 0)
    s["oblig_acc"] = round(max(0.0, 1.0 - abs(gt_o - got_o) / max(1, gt_o)), 3)
    return s

def row_from_global(kind_candidate: str, kind: str, gt_doc: dict,
                    got: dict, dt: float, error: str = None) -> dict:
    gt  = gt_doc.get("global", {})
    row = {
        "candidate":     kind_candidate,
        "kind":          kind,
        "field_accuracy": float("nan"),
        "table_accuracy": float("nan"),
        "structure_score": float("nan"),
        "obligation_match": float("nan"),
        "processing_time_s": round(dt, 2),
        "error_rate": 1 if error else 0,
    }
    if error:
        row["error_msg"] = str(error)[:200]
        return row
    s = score_global(gt, got)
    row["field_accuracy"]  = round((s["title_ok"] + s["instrument_ok"] + s["date_ok"]) / 3, 3)
    row["table_accuracy"]  = round(1.0 - s["table_acc"], 3)
    row["structure_score"] = s["structure"]
    row["obligation_match"] = s["oblig_acc"]
    return row

rprint("[green]evaluator ready[/green]")


evaluator ready

## 5   Run the evaluation

`EVAL_ALL=True` evaluates all 25 docs. LLM prompts are built once per
document and shared by both models (same input, fair comparison), and every
call hits the `llm_cache` so a re-run is nearly free.

Text rows: the parser's own markdown is scored directly. LLM rows: the model
returns a JSON structure on the same text; unparseable output is an error row.


In [5]:


MAX_CHARS = 48000   # prompt input budget: front matter + early articles

def llm_row(doc_id: str, gt_doc: dict, model: str, prompt: str) -> dict:
    ck = c.stable_hash(model, prompt)
    t0 = _time.time()
    cached = (c.CACHE_PATH / f"{model.replace(':', '_')}_{ck}.json").exists()
    try:
        raw = c.ollama_chat(model, prompt, temperature=0.0,
                            num_ctx=32768,
                            num_predict=24576 if "deepseek" in model else 8192,
                            max_retries=3, use_cache=True, cache_key=ck)
        got = c.extract_json(raw)
        dt = 0.0 if cached else _time.time() - t0
        return row_from_global(model, "llm", gt_doc, got, dt)
    except Exception as e:
        return row_from_global(model, "llm", gt_doc, None, _time.time() - t0,
                               error=f"{type(e).__name__}: {str(e)[:120]}")

def content_to_got(content: str) -> dict:
    """detect the structural fields from any converted text (text-parser rows)."""
    return {
        "document_title":   GT._extract_title_heuristic(content[:6000]),
        "instrument":       GT._instrument(content[:8000]),
        "publication_date": GT._publication_date(content[:8000]),
        "tables_count":     GT._count_tables(content),
        "articles":         [{"number": a} for a in GT._extract_article_numbers(content)],
    }

rows = []
sel  = selected_docs()
pdf_by_doc = {}
for pdf in doc_list():
    pdf_by_doc[pdf.stem] = pdf

for doc_id, gt_doc in track(sel.items(), description="evaluating"):
    rprint("[bold]%s[/bold]" % doc_id)
    pdf = pdf_by_doc.get(doc_id)
    if pdf is None:
        rprint("  [yellow]no PDF on disk, skipping[/yellow]")
        continue

    # text parsers: pymupdf4llm reads the on-disk markdown (already produced
    # in cell 2); docling/unstructured reconvert the PDF live
    for parser_name in TEXT_PARSERS:
        t0 = _time.time()
        try:
            if parser_name == "pymupdf4llm":
                md_path = c.md_for_pdf(pdf)
                content = md_path.read_text() if md_path.exists() else P.PARSERS[parser_name](pdf)
            else:
                content = P.PARSERS[parser_name](pdf)
            dt = _time.time() - t0
            got = content_to_got(content)
            row = row_from_global(parser_name, "text", gt_doc, got, dt)
        except Exception as e:
            dt = _time.time() - t0
            row = row_from_global(parser_name, "text", gt_doc, None, dt,
                                  error=f"{type(e).__name__}: {str(e)[:120]}")
        rows.append({"doc": doc_id, **row})
        rprint("   %-15s field=%.3f table=%.3f struct=%.3f t=%.1fs err=%d" %
               (parser_name, row["field_accuracy"], row["table_accuracy"],
                row["structure_score"], dt, row["error_rate"]))

    # LLM rows: one shared prompt for both models
    base_md = c.md_for_pdf(pdf)
    text = base_md.read_text() if base_md.exists() else ""
    if not text and pdf.exists():
        text = P.PARSERS["pymupdf4llm"](pdf)
    prompt = LEX.STRUCTURE_PROMPT + "\n===== TEXT =====\n" + text[:MAX_CHARS]
    for model in LLM_MODELS:
        row = llm_row(doc_id, gt_doc, model, prompt)
        rows.append({"doc": doc_id, **row})
        rprint("   %-20s field=%.3f table=%.3f struct=%.3f obl=%.3f t=%.1fs err=%d" %
               (model, row["field_accuracy"], row["table_accuracy"],
                row["structure_score"], row["obligation_match"],
                row["processing_time_s"], row["error_rate"]))

df = pd.DataFrame(rows)
df["composite"] = df[["field_accuracy", "table_accuracy", "structure_score"]].mean(axis=1).round(3)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_csv = c.EVALUATIONS_PATH / f"parser_evaluation_{ts}.csv"
c.EVALUATIONS_PATH.mkdir(parents=True, exist_ok=True)
df.to_csv(out_csv, index=False)
rprint(f"\n[green]evaluated {len(df)} rows -> {out_csv}[/green]")
console.print(df.to_string(index=False))


Output()

elec_dir_2019_944

pymupdf4llm     field=0.667 table=1.000 struct=1.000 t=0.0s err=0

[INFO] 2026-08-29 21:20:21,616 [RapidOCR] base.py:22: Using engine_name: onnxruntime

[INFO] 2026-08-29 21:20:21,646 [RapidOCR] download_file.py:60: File exists and is valid: 
/home/iauser/.pyenv/versions/3.11.9/envs/energy-audit/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_
mobile.onnx

...

[INFO] 2026-08-29 21:29:37,698 [RapidOCR] download_file.py:60: File exists and is valid: 
/home/iauser/.pyenv/versions/3.11.9/envs/energy-audit/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_
v2.0_cls_mobile.onnx

[INFO] 2026-08-29 21:29:37,699 [RapidOCR] main.py:57: Using 
/home/iauser/.pyenv/versions/3.11.9/envs/energy-audit/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_
v2.0_cls_mobile.onnx

[INFO] 2026-08-29 21:29:37,743 [RapidOCR] base.py:22: Using engine_name: onnxruntime

[INFO] 2026-08-29 21:29:37,765 [RapidOCR] download_file.py:60: File exists and is valid: 
/home/iauser/.pyenv/versions/3.11.9/envs/energy-audit/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_rec_
mobile.onnx

[INFO] 2026-08-29 21:29:37,766 [RapidOCR] main.py:57: Using 
/home/iauser/.pyenv/versions/3.11.9/envs/energy-audit/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_rec_
mobile.onnx

## 6   Report & hand-off
Per-candidate mean composite; the best **text parser** (best LLM model is
tracked separately since 03b will call one) is written to
`notebooks/data/evaluations/best_parser.json` for the structure-extraction
notebook.

In [6]:

summary = (df.groupby(["candidate", "kind"])
             .agg(mean_field=("field_accuracy", "mean"),
                  mean_table=("table_accuracy", "mean"),
                  mean_struct=("structure_score", "mean"),
                  composite=("composite", "mean"),
                  errors=("error_rate", "sum"),
                  avg_time_s=("processing_time_s", "mean"))
             .reset_index()
             .sort_values("composite", ascending=False))
summary["mean_field"]  = summary["mean_field"].round(3)
summary["mean_table"]  = summary["mean_table"].round(3)
summary["mean_struct"] = summary["mean_struct"].round(3)
summary["composite"]   = summary["composite"].round(3)
summary["avg_time_s"]  = summary["avg_time_s"].round(1)
console.print(summary.to_string(index=False))

best_text = summary[summary["kind"] == "text"].sort_values("composite", ascending=False).head(1).to_dict("records")[0]
best_llm  = summary[summary["kind"] == "llm"].sort_values("composite", ascending=False).head(1).to_dict("records")[0]

handoff = {
    "generated": datetime.now().isoformat(),
    "docs_evaluated": df["doc"].nunique(),
    "best_text_parser": {
        "name": best_text["candidate"],
        "composite": best_text["composite"],
        "mean_field": best_text["mean_field"],
        "mean_table": best_text["mean_table"],
        "mean_struct": best_text["mean_struct"],
        "avg_time_s": best_text["avg_time_s"],
        "errors": int(best_text["errors"]),
    },
    "best_llm_extractor": {
        "name": best_llm["candidate"],
        "composite": best_llm["composite"],
        "mean_field": best_llm["mean_field"],
        "mean_table": best_llm["mean_table"],
        "mean_struct": best_llm["mean_struct"],
        "mean_obligation": round(float(df[df["candidate"] == best_llm["candidate"]]["obligation_match"].mean()), 3),
        "avg_time_s": best_llm["avg_time_s"],
        "errors": int(best_llm["errors"]),
    },
    "full_summary": summary.to_dict("records"),
}
out_json = c.EVALUATIONS_PATH / "best_parser.json"
with open(out_json, "w") as f:
    json.dump(handoff, f, indent=2)
rprint(f"\n[bold green]best text parser :[/bold green] {best_text['candidate']} (composite {best_text['composite']})")
rprint(f"[bold green]best llm extractor :[/bold green] {best_llm['candidate']} (composite {best_llm['composite']})")
rprint(f"[green]hand-off written -> {out_json}[/green]")


candidate kind  mean_field  mean_table  mean_struct  composite  errors  avg_time_s
    pymupdf4llm text       0.640       1.000        1.000      0.880       0         0.0
   unstructured text       0.614       0.541        0.691      0.616       0         5.5
deepseek-r1:32b  llm       0.972       0.514        0.229      0.572       1         0.0
        docling text       0.534       0.474        0.688      0.565       0        26.3
    qwen3.8:27b  llm       0.905       0.581        0.160      0.549       4         0.0

best text parser : pymupdf4llm (composite 0.88)

best llm extractor : deepseek-r1:32b (composite 0.572)

hand-off written -> /home/iauser/AIProjects/energy-audit/notebooks/data/evaluations/best_parser.json